# Baseline Training Notebook
🔹 Goal: Fine-tune a RoBERTa model on SemEval-2024 Task 8 Subtask A (binary classification: human-written vs. machine-generated text).
🔹 Output: A trained RoBERTa model saved as "baseline_roberta.pth".

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1: Setup & Install Dependencies

In [ ]:
%pip install transformers jsonlines tqdm

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import jsonlines
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from tqdm import tqdm
from sklearn.metrics import f1_score

## Step 2: Load & Preprocess Dataset

### 🔹 2.1 Define Custom Dataset Class

In [ ]:
class TextDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=512):
        self.texts = []
        self.labels = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Load JSONL file
        with jsonlines.open(file_path) as reader:
            for obj in reader:
                self.texts.append(obj["text"])
                self.labels.append(obj["label"])

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

### 🔹 2.2 Load Train & Dev Datasets

In [ ]:
# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Define paths
train_file = "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_train_monolingual.jsonl"
dev_file = "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_dev_monolingual.jsonl"

# Create datasets
train_dataset = TextDataset(train_file, tokenizer)
dev_dataset = TextDataset(dev_file, tokenizer)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

## Step 3: Load Pretrained RoBERTa Model

In [ ]:
# Load RoBERTa model with classification head
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
model.to("cuda")  # Move to GPU if available

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

## Step 4: Define Optimizer & Loss Function

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

## Step 5: Train the Model

### 🔹 5.1 Define Training Function

In [ ]:
def train_and_validate(model, train_loader, dev_loader, optimizer, criterion, device="cuda", epochs=3):
    model.to(device)

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        # Training phase
        model.train()
        total_train_loss = 0.0

        progress_bar = tqdm(train_loader, desc="Training", leave=False)

        for batch in progress_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask).logits
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

            # Update progress bar with current batch loss
            progress_bar.set_postfix({"Batch Loss": loss.item()})

        avg_train_loss = total_train_loss / len(train_loader)
        print(f"→ Avg Train Loss: {avg_train_loss:.4f}")

        # Validation phase
        model.eval()
        total_dev_loss = 0.0
        all_preds = []
        all_labels = []

        progress_bar = tqdm(dev_loader, desc="Validating", leave=False)

        with torch.no_grad():
            for batch in progress_bar:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = model(input_ids, attention_mask=attention_mask).logits
                loss = criterion(outputs, labels)

                total_dev_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            macro_f1 = f1_score(all_labels, all_preds, average="macro")

        avg_dev_loss = total_dev_loss / len(dev_loader)
        print(f"→ Avg Dev Loss: {avg_dev_loss:.4f}, Macro F1: {macro_f1:.4f}")

### 🔹 5.2 Train the Model

In [ ]:
train_and_validate(model, train_loader, dev_loader, optimizer, criterion, epochs=3)


Epoch 1/3


→ Avg Train Loss: 0.0243


→ Avg Dev Loss: 1.2716, Macro F1: 0.7080

Epoch 2/3


→ Avg Train Loss: 0.0093


→ Avg Dev Loss: 2.2171, Macro F1: 0.6085

Epoch 3/3


→ Avg Train Loss: 0.0071


→ Avg Dev Loss: 2.0452, Macro F1: 0.6556


## Step 6: Save the Trained Model

In [ ]:
# Set directory for saving baseline model
baseline_model_path = "/content/drive/MyDrive/SemEval-2024 Task 8/models/baseline"
os.makedirs(baseline_model_path, exist_ok=True)

# Save model checkpoint
model_save_path = os.path.join(baseline_model_path, "baseline_roberta.pth")
torch.save(model.state_dict(), model_save_path)

print(f"Baseline model saved at: {model_save_path}")

Baseline model saved at: /content/drive/MyDrive/SemEval-2024 Task 8/models/baseline/baseline_roberta.pth
